#  SkinSight AI — 06 · Démo Webcam & AR

Ce notebook documente la fonctionnalité webcam temps réel avec overlay AR.
L'overlay détecte les zones anatomiques du visage (front, joues, nez, menton)
et applique une analyse par zone.

**Stack :** OpenCV · MediaPipe Face Mesh · Canvas overlay

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import os, warnings
warnings.filterwarnings('ignore')

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))

print(f'OpenCV version : {cv2.__version__}')
try:
    import mediapipe as mp
    print(f'MediaPipe version : {mp.__version__}')
except ImportError:
    print('  MediaPipe non installé — pip install mediapipe')

OpenCV version : 4.13.0
MediaPipe version : 0.10.35


## 1. Architecture de la fonctionnalité webcam AR

In [7]:
print("""
┌─────────────────────────────────────────────────────────────────┐
│              Pipeline Webcam AR — SkinSight AI                  │
│                                                                 │
│  Webcam (getUserMedia)                                          │
│       │                                                         │
│       ▼                                                         │
│  <video> HTML5 ──► <canvas> overlay                            │
│       │                                                         │
│       ▼ (frame toutes les 100ms)                                │
│  MediaPipe Face Mesh (468 landmarks)                            │
│       │                                                         │
│       ▼                                                         │
│  Détection zones :                                              │
│    • Front      (landmarks 10, 67, 109...)                      │
│    • Joue gauche (landmarks 234, 93...)                         │
│    • Joue droite (landmarks 454, 323...)                        │
│    • Nez        (landmarks 1, 2, 4...)                          │
│    • Menton     (landmarks 18, 200...)                          │
│       │                                                         │
│       ▼                                                         │
│  Crop zone ──► POST /predict ──► Overlay coloré + label        │
│                                                                 │
│  Légende couleurs :                                             │
│    🟢 Peau saine        🟡 Acné légère                          │
│    🔴 Acné inflamm.     🟣 Rosacée    🟤 Hyperpigm.             │
└─────────────────────────────────────────────────────────────────┘
""")


┌─────────────────────────────────────────────────────────────────┐
│              Pipeline Webcam AR — SkinSight AI                  │
│                                                                 │
│  Webcam (getUserMedia)                                          │
│       │                                                         │
│       ▼                                                         │
│  <video> HTML5 ──► <canvas> overlay                            │
│       │                                                         │
│       ▼ (frame toutes les 100ms)                                │
│  MediaPipe Face Mesh (468 landmarks)                            │
│       │                                                         │
│       ▼                                                         │
│  Détection zones :                                              │
│    • Front      (landmarks 10, 67, 109...)                      │
│    • Joue gauche (landmarks 234, 93...)       

## 2. Détection de visage avec MediaPipe

In [13]:
import mediapipe as mp
print(mp.__version__)

0.10.35


In [15]:
try:
    import mediapipe as mp
    print(f'MediaPipe version : {mp.__version__}')

    # Nouvelle API MediaPipe 0.10+
    ZONES = {
        'Front':       [10, 67, 109, 338, 297, 332],
        'Joue gauche': [234, 93, 132, 58, 172],
        'Joue droite': [454, 323, 361, 288, 397],
        'Nez':         [1, 2, 4, 5, 6, 19, 20],
        'Menton':      [18, 200, 199, 175, 152],
    }

    ZONE_COLORS = {
        'Front':       (100, 180, 100),
        'Joue gauche': (180, 100, 100),
        'Joue droite': (180, 100, 100),
        'Nez':         (100, 100, 180),
        'Menton':      (180, 160, 100),
    }

    print('Zones configurees :', list(ZONES.keys()))

except ImportError:
    print('MediaPipe non installe — pip install mediapipe')

MediaPipe version : 0.10.35
Zones configurees : ['Front', 'Joue gauche', 'Joue droite', 'Nez', 'Menton']


## 3. Démonstration sur une image statique

In [19]:
try:
    import mediapipe as mp
    import glob

    # Trouver une image de visage dans le dataset
    test_imgs = []
    for cls in ['saine', 'acne_inflammatoire']:
        folder = os.path.join(BASE_DIR, 'data', 'val', cls)
        if os.path.exists(folder):
            imgs = glob.glob(os.path.join(folder, '*.jpg'))
            if imgs: test_imgs.append(imgs[0])

    if not test_imgs:
        print('Aucune image trouvée dans data/val/ — test ignoré')
    else:
        img_path = test_imgs[0]
        img_bgr  = cv2.imread(img_path)
        img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w     = img_rgb.shape[:2]

        with mp.solutions.face_mesh.FaceMesh(
            static_image_mode=True,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.5
        ) as face_mesh:
            results = face_mesh.process(img_rgb)

        overlay = img_rgb.copy()

        if results.multi_face_landmarks:
            lms = results.multi_face_landmarks[0].landmark
            for zone_name, zone_ids in ZONES.items():
                pts = np.array([[int(lms[i].x * w), int(lms[i].y * h)] for i in zone_ids])
                color = ZONE_COLORS[zone_name]
                cv2.fillPoly(overlay, [pts], color)
                cx, cy = pts.mean(axis=0).astype(int)
                cv2.putText(overlay, zone_name, (cx-30, cy),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1, cv2.LINE_AA)
            blended = cv2.addWeighted(img_rgb, 0.6, overlay, 0.4, 0)
            print(f' Visage détecté — {len(results.multi_face_landmarks)} visage(s)')
        else:
            blended = img_rgb
            print('  Aucun visage détecté dans cette image')

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].imshow(img_rgb);  axes[0].set_title('Image originale'); axes[0].axis('off')
        axes[1].imshow(blended);  axes[1].set_title('Overlay zones AR'); axes[1].axis('off')

        patches = [mpatches.Patch(color=np.array(c)/255, label=n) for n, c in ZONE_COLORS.items()]
        axes[1].legend(handles=patches, loc='lower right', fontsize=8)

        plt.suptitle('Détection zones anatomiques — MediaPipe Face Mesh', fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(BASE_DIR, 'notebooks', 'fig_06_ar_zones.png'), dpi=150, bbox_inches='tight')
        plt.show()

except ImportError:
    print('MediaPipe non installé — démonstration ignorée')

Aucune image trouvée dans data/val/ — test ignoré


## 4. Code JavaScript de la page Webcam AR

La page webcam de `app/index.html` implémente le pipeline suivant :

In [1]:
# Extrait du code JS de la page webcam (documenté ici pour référence)
js_doc = """
// 1. Accès webcam
const stream = await navigator.mediaDevices.getUserMedia({ video: true });
video.srcObject = stream;

// 2. Initialisation MediaPipe Face Mesh
const faceMesh = new FaceMesh({ locateFile: ... });
faceMesh.onResults(onResults);

// 3. Callback résultats — dessin des zones
function onResults(results) {
    ctx.clearRect(0, 0, canvas.width, canvas.height);
    ctx.drawImage(results.image, 0, 0);
    if (results.multiFaceLandmarks) {
        for (const landmarks of results.multiFaceLandmarks) {
            drawZones(ctx, landmarks, canvas.width, canvas.height);
        }
    }
}

// 4. Analyse périodique (toutes les 3s)
setInterval(async () => {
    const blob = await captureFrame(canvas);
    const result = await fetch('/predict', { method:'POST', body: formData(blob) });
    displayOverlay(result);
}, 3000);
"""
print(js_doc)


// 1. Accès webcam
const stream = await navigator.mediaDevices.getUserMedia({ video: true });
video.srcObject = stream;

// 2. Initialisation MediaPipe Face Mesh
const faceMesh = new FaceMesh({ locateFile: ... });
faceMesh.onResults(onResults);

// 3. Callback résultats — dessin des zones
function onResults(results) {
    ctx.clearRect(0, 0, canvas.width, canvas.height);
    ctx.drawImage(results.image, 0, 0);
    if (results.multiFaceLandmarks) {
        for (const landmarks of results.multiFaceLandmarks) {
            drawZones(ctx, landmarks, canvas.width, canvas.height);
        }
    }
}

// 4. Analyse périodique (toutes les 3s)
setInterval(async () => {
    const blob = await captureFrame(canvas);
    const result = await fetch('/predict', { method:'POST', body: formData(blob) });
    displayOverlay(result);
}, 3000);



## 5. Lancer la démo webcam

In [3]:
import webbrowser, subprocess, sys

print('Pour lancer la démo webcam AR :')
print()
print('  Terminal 1 — API FastAPI')
print('  conda activate skinsight')
print('  cd C:\\Users\\nasri\\OneDrive\\Desktop\\Projects\\SkinSightAI')
print('  uvicorn app.api.main:app --reload --port 8000')
print()
print('  Terminal 2 — Ouvrir le frontend')
print('  Double-clic sur app/index.html')
print('  Aller sur la page "Webcam AR" dans la sidebar')
print()
print('  Autoriser l\'accès caméra dans le navigateur')
print('  L\'overlay s\'affiche automatiquement sur le flux vidéo')

Pour lancer la démo webcam AR :

  Terminal 1 — API FastAPI
  conda activate skinsight
  cd C:\Users\nasri\OneDrive\Desktop\Projects\SkinSightAI
  uvicorn app.api.main:app --reload --port 8000

  Terminal 2 — Ouvrir le frontend
  Double-clic sur app/index.html
  Aller sur la page "Webcam AR" dans la sidebar

  Autoriser l'accès caméra dans le navigateur
  L'overlay s'affiche automatiquement sur le flux vidéo


##  Résumé

| Composant | Technologie |
|-----------|-------------|
| Capture vidéo | `navigator.mediaDevices.getUserMedia()` |
| Détection landmarks | MediaPipe Face Mesh (468 points) |
| Overlay zones | HTML5 Canvas API |
| Analyse par zone | Crop → `POST /predict` → JSON |
| Fréquence d'analyse | 1 frame toutes les 3 secondes |
| Zones détectées | Front · Joues (×2) · Nez · Menton |

> **Note :** La précision de l'analyse par zone dépend de la qualité de la webcam
> et de la taille du crop. Une zone trop petite peut réduire la fiabilité du diagnostic.